In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
#takes ~5 min
!apt-get install -y aria2 p7zip-full

!aria2c -x 16 -s 16 -k 1M -q -o msmd.zip.part \
"https://zenodo.org/record/2597505/files/msmd_aug_v1-1_no-audio.zip"

!7z x msmd.zip.part -o/content/msmd_dataset -mmt=on

In [ ]:
!pip install muscima

In [ ]:
import os
import random
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchaudio
import torchaudio.transforms as T_audio
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models import swin_v2_t, Swin_V2_T_Weights
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import muscima
import muscima.io
import copy
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.nn.utils import clip_grad_norm_
hf_hub_download(repo_id="hyg444/SheetAudio-GCL", filename="models.py", local_dir=".")
from models import SheetMusicSwin, SpectrogramSwin, VisionAudioMoCo


###Data + utils


In [ ]:
class MSMDSpectrogramPipeline(torch.nn.Module):
    def __init__(self, filterbank):
        super().__init__()
        self.n_fft = 2048
        self.hop_length = 1102
        self.register_buffer('filterbank', filterbank)
        self.spectrogram = torchaudio.transforms.Spectrogram(
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            power=1.0
        )

    def forward(self, waveform):
        mag_spec = self.spectrogram(waveform)
        filtered_spec = torch.einsum('bft,fm->bmt', mag_spec, self.filterbank)

        log_spec = torch.log10(filtered_spec + 1.0)

        return log_spec

In [ ]:
def letterbox_image(pil_img, target_w=416, target_h=128, pad_color=(255, 255, 255)):
    orig_w, orig_h = pil_img.size
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)
    resized = pil_img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (target_w, target_h), color=pad_color)
    paste_x = (target_w - new_w) // 2
    paste_y = (target_h - new_h) // 2
    canvas.paste(resized, (paste_x, paste_y))
    return canvas

def get_deterministic_splits(root_dir, train_ratio=0.75, val_ratio=0.10):
    pieces = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])
    rng = random.Random(42)
    rng.shuffle(pieces)
    n = len(pieces)
    train_end = int(train_ratio * n)
    val_end = train_end + int(val_ratio * n)
    return pieces[:train_end], pieces[train_end:val_end], pieces[val_end:]

def eval_collate_fn(batch):
    images = torch.stack([item['image'] for item in batch])
    line_ids = [item['line_id'] for item in batch]

    spectrograms = [item['spectrogram'] for item in batch]

    #  200 frames (10 seconds)
    target_frames = 200
    padded_spectrograms = []

    for spec in spectrograms:
        seq_len = spec.shape[2]
        # Pad to exactly 200 if under
        pad_amount = max(0, target_frames - seq_len)
        padded_spec = F.pad(spec, (0, pad_amount))
        padded_spectrograms.append(padded_spec)

    return {
        'images': images,
        'spectrograms': torch.stack(padded_spectrograms),
        'line_id': line_ids
    }

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

In [ ]:
class MSMDNoGraphDataset(Dataset):
    def __init__(self, root_dir, split_pieces, transform=None):
        self.root_dir = root_dir
        self.transform = transform or T.Compose([T.ToTensor()])
        self.val_frames = 200 # 10 seconds
        self.data_index = []

        print(f"Indexing MSMD: {len(split_pieces)} pieces")
        for piece_name in tqdm(split_pieces, desc="Parsing MSMD Validation/Test"):
            piece_dir = os.path.join(root_dir, piece_name)
            if not os.path.isdir(piece_dir): continue

            perf_dir = os.path.join(piece_dir, 'performances')
            performances = [p for p in os.listdir(perf_dir) if os.path.isdir(os.path.join(perf_dir, p))]

            score_dir_base = os.path.join(piece_dir, 'scores')
            score_folders = os.listdir(score_dir_base)
            if not score_folders: continue
            score_path = os.path.join(score_dir_base, score_folders[0])
            xml_files = sorted(glob.glob(os.path.join(score_path, 'mung', '*.xml')))

            for xml_file in xml_files:
                page_id = os.path.basename(xml_file).replace('.xml', '')
                systems_npy_path = os.path.join(score_path, 'coords', f"systems_{page_id}.npy")
                if not os.path.exists(systems_npy_path): continue

                img_path = os.path.join(score_path, 'img', f"{page_id}.png")
                system_boxes = np.load(systems_npy_path)
                nodes = muscima.io.parse_cropobject_list(xml_file)

                for perf_name in performances:
                    spec_path = os.path.join(perf_dir, perf_name, 'features', f"{perf_name}.flac_spec.npy")
                    notes_path = os.path.join(perf_dir, perf_name, 'features', f"{perf_name}.flac_notes.npy")

                    if not os.path.exists(notes_path) or not os.path.exists(spec_path):
                        continue

                    notes_array = np.load(notes_path)
                    onset_key = f"{perf_name}_onset_frame"
                    event_idx_key = f"{perf_name}_note_event_idx"

                    for box in system_boxes:
                        if np.ptp(box[:, 0]) < np.ptp(box[:, 1]):
                            sys_top, sys_bottom = float(np.min(box[:, 0])), float(np.max(box[:, 0]))
                        else:
                            sys_top, sys_bottom = float(np.min(box[:, 1])), float(np.max(box[:, 1]))

                        crop_top = max(0, sys_top - 30)
                        crop_bottom = sys_bottom + 30
                        line_id = f"{piece_name}_{page_id}_{sys_top}"

                        valid_nodes = [
                            n for n in nodes
                            if crop_top <= n.top <= crop_bottom
                            and n.clsname not in ['system', 'staff', 'measure', 'staff_space']
                        ]
                        if not valid_nodes: continue

                        min_start_frame = float('inf')
                        max_end_frame = 0

                        for n in valid_nodes:
                            if n.clsname == 'notehead-full' and onset_key in n.data:
                                onset_frame = int(n.data[onset_key])
                                min_start_frame = min(min_start_frame, onset_frame)
                                event_idx = n.data.get(event_idx_key)
                                if event_idx is not None and event_idx < len(notes_array):
                                    duration_frames = int(float(notes_array[event_idx, 2]) * 20.0)
                                    max_end_frame = max(max_end_frame, onset_frame + duration_frames)
                                else:
                                    max_end_frame = max(max_end_frame, onset_frame + 20)

                        if min_start_frame == float('inf'):
                            continue

                        self.data_index.append({
                            'img_path': img_path,
                            'spec_path': spec_path,
                            'crop_top': int(crop_top),
                            'crop_bottom': int(crop_bottom),
                            'start_frame': int(min_start_frame),
                            'end_frame': int(max_end_frame) + 5,
                            'line_id': line_id
                        })

    def __len__(self):
        return len(self.data_index)

    def __getitem__(self, idx):
        record = self.data_index[idx]

        start_frame = record['start_frame']
        end_frame = record['end_frame']
        total_frames = end_frame - start_frame

        full_spec = np.load(record['spec_path'], mmap_mode='r')

        if total_frames <= self.val_frames:
            spec_slice = full_spec[:, start_frame : end_frame]
        else:
            center_point = total_frames // 2
            half_crop = self.val_frames // 2
            s_idx = start_frame + center_point - half_crop
            e_idx = s_idx + self.val_frames
            spec_slice = full_spec[:, s_idx : e_idx]

        spec_slice = (spec_slice - np.mean(spec_slice)) / (np.std(spec_slice) + 1e-6)
        spec_tensor = torch.tensor(np.copy(spec_slice), dtype=torch.float32).unsqueeze(0)

        with Image.open(record['img_path']).convert('RGB') as img:
            img_width, img_height = img.size
            crop_bottom = min(record['crop_bottom'], img_height)
            crop_img = img.crop((0, record['crop_top'], img_width, crop_bottom))

            letterboxed_img = letterbox_image(crop_img, target_w=416, target_h=128)
            img_tensor = self.transform(letterboxed_img)

        return {
            'image': img_tensor,
            'spectrogram': spec_tensor,
            'line_id': record['line_id']
        }

In [ ]:
@torch.no_grad()
def evaluate_retrieval(vision_encoder, audio_encoder, val_loader, device='cuda'):
    vision_encoder.eval()
    audio_encoder.eval()

    all_vision_embeddings = []
    all_audio_embeddings = []
    all_line_ids = []

    for batch_data in tqdm(val_loader, desc="Evaluation Forward Pass"):
        images = batch_data['images'].to(device)
        audio_inputs = batch_data['spectrograms'].to(device)

        q_audio = audio_encoder(audio_inputs)
        q_vision = vision_encoder(images)

        all_vision_embeddings.append(q_vision.cpu())
        all_audio_embeddings.append(q_audio.cpu())
        all_line_ids.extend(batch_data['line_id'])

    vision_embeds = torch.cat(all_vision_embeddings, dim=0)
    audio_embeds = torch.cat(all_audio_embeddings, dim=0)

    unique_line_ids = []
    unique_vision_indices = []
    seen = set()
    for idx, lid in enumerate(all_line_ids):
        if lid not in seen:
            seen.add(lid)
            unique_line_ids.append(lid)
            unique_vision_indices.append(idx)

    dedup_vision_embeds = vision_embeds[unique_vision_indices]

    similarity_matrix = torch.matmul(audio_embeds, dedup_vision_embeds.T)

    encoder = LabelEncoder()
    encoder.fit(unique_line_ids)

    query_labels = torch.tensor(encoder.transform(all_line_ids))
    gallery_labels = torch.tensor(encoder.transform(unique_line_ids))

    ground_truth_mask = (query_labels.unsqueeze(1) == gallery_labels.unsqueeze(0))

    def calculate_ranks(sim_matrix, mask):
        sorted_indices = torch.argsort(sim_matrix, dim=1, descending=True)
        sorted_mask = torch.gather(mask, 1, sorted_indices)

        ranks = sorted_mask.float().argmax(dim=1) + 1
        ranks = ranks.float()

        r1 = (ranks <= 1).float().mean().item()
        r25 = (ranks <= 25).float().mean().item()
        mrr = (1.0 / ranks).mean().item()
        mr = ranks.median().item()
        return r1, r25, mrr, mr

    metrics = {}
    r1, r25, mrr, mr = calculate_ranks(similarity_matrix, ground_truth_mask)
    metrics['A2V'] = {'R@1': r1, 'R@25': r25, 'MRR': mrr, 'MR': mr}

    r1, r25, mrr, mr = calculate_ranks(similarity_matrix.T, ground_truth_mask.T)
    metrics['V2A'] = {'R@1': r1, 'R@25': r25, 'MRR': mrr, 'MR': mr}

    return metrics

###Evaluate on MSMD test split

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

audio_path = hf_hub_download(repo_id="hyg444/SheetAudio-GCL", filename="SheetAudio-GCL-Audio.pth")
vision_path = hf_hub_download(repo_id="hyg444/SheetAudio-GCL", filename="SheetAudio-GCL-Sheet.pth")
filterbank_path = hf_hub_download(repo_id="hyg444/SheetAudio-GCL", filename="madmom_filterbank.pt")

vision_encoder = SheetMusicSwin().to(device)
audio_encoder = SpectrogramSwin().to(device)

vision_encoder.load_state_dict(torch.load(vision_path, map_location=device))
audio_encoder.load_state_dict(torch.load(audio_path, map_location=device))

vision_encoder.eval()
audio_encoder.eval()

BATCH_SIZE = 64

DATASET_ROOT = '/content/msmd_dataset/msmd_aug_v1-1_no-audio'
if os.path.exists(DATASET_ROOT):
    _, _, test_pieces = get_deterministic_splits(DATASET_ROOT)

    msmd_test_dataset = MSMDNoGraphDataset(root_dir=DATASET_ROOT, split_pieces=test_pieces)
    msmd_test_loader = DataLoader(
        msmd_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=eval_collate_fn,
        num_workers=2
    )

    print("\n--- Evaluating on MSMD Test Split ---")
    msmd_metrics = evaluate_retrieval(vision_encoder, audio_encoder, msmd_test_loader, device)
    print("Audio-to-Vision:")
    print(f"  R@1: {msmd_metrics['A2V']['R@1']:.3f} | R@25: {msmd_metrics['A2V']['R@25']:.3f} | MRR: {msmd_metrics['A2V']['MRR']:.3f} | MR: {msmd_metrics['A2V']['MR']}")
    print("Vision-to-Audio:")
    print(f"  R@1: {msmd_metrics['V2A']['R@1']:.3f} | R@25: {msmd_metrics['V2A']['R@25']:.3f} | MRR: {msmd_metrics['V2A']['MRR']:.3f} | MR: {msmd_metrics['V2A']['MR']}")
else:
    print(f"\nMSMD root '{DATASET_ROOT}' not found. Skipping MSMD evaluation.")

###Fine tune on Grandstaff


In [ ]:
class GrandstaffFinetuneDataset(Dataset):
    def __init__(self, hf_dataset, audio_pipeline, transform=None, mode='train'):
        self.hf_dataset = hf_dataset
        self.audio_pipeline = audio_pipeline
        self.transform = transform or T.Compose([T.ToTensor()])
        self.mode = mode

        self.min_frames = 80   # 4 seconds
        self.max_frames = 160  # 8 seconds
        self.val_frames = 200  # 10 seconds

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]

        img = item['image'].convert('RGB')
        img_letterboxed = letterbox_image(img, 416, 128)
        img_tensor = self.transform(img_letterboxed)

        audio_array = item['audio']['array']
        sr = item['audio']['sampling_rate']
        waveform = torch.tensor(audio_array).float()

        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)

        if sr != 22050:
            resampler = T_audio.Resample(orig_freq=sr, new_freq=22050)
            waveform = resampler(waveform)

        with torch.no_grad():
            spectrogram = self.audio_pipeline(waveform)

        total_frames = spectrogram.shape[2]

        if self.mode == 'train':
            if total_frames > self.min_frames:
                crop_len = random.randint(self.min_frames, min(self.max_frames, total_frames))
                max_start = total_frames - crop_len
                s_idx = random.randint(0, max_start)
                e_idx = s_idx + crop_len
                spectrogram = spectrogram[:, :, s_idx:e_idx]
        else:
            # 10-second center crop for evaluation
            if total_frames > self.val_frames:
                center_point = total_frames // 2
                half_crop = self.val_frames // 2
                s_idx = center_point - half_crop
                e_idx = s_idx + self.val_frames
                spectrogram = spectrogram[:, :, s_idx:e_idx]

        spectrogram = (spectrogram - spectrogram.mean()) / (spectrogram.std() + 1e-6)

        return {
            'images': img_tensor,
            'spectrograms': spectrogram,
            'line_id': f"grandstaff_{idx}"
        }

def grandstaff_collate_fn(batch):
    images = torch.stack([item['images'] for item in batch])
    line_ids = [item['line_id'] for item in batch]

    spectrograms = [item['spectrograms'] for item in batch]

    target_frames = 200
    padded_spectrograms = []

    for spec in spectrograms:
        seq_len = spec.shape[2]
        pad_amount = max(0, target_frames - seq_len)
        padded_spec = F.pad(spec, (0, pad_amount))
        padded_spectrograms.append(padded_spec)

    return {
        'images': images,
        'spectrograms': torch.stack(padded_spectrograms),
        'line_id': line_ids
    }

def load_checkpoint(filepath, model, optimizer=None, device='cuda'):
    """Helper to load checkpoint if you don't already have it defined."""
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', None)
    return model, optimizer, epoch, loss, checkpoint

In [ ]:
def train_phase_3(
    train_loader,
    val_loader,
    vision_student,
    audio_teacher,
    num_epochs=30,
    learning_rate=5e-4,
    save_dir='/content/drive/MyDrive/MSMD_Checkpoints',
    resume_checkpoint='/content/drive/MyDrive/MSMD_Checkpoints/grandstaff_finetune_best.pth',
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    print(f"Starting Phase 3 Joint MoCo Fine-Tuning on: {device}")
    os.makedirs(save_dir, exist_ok=True)

    moco_model = VisionAudioMoCo(
        vision_encoder=vision_student,
        audio_encoder=audio_teacher,
        dim=512,
        K=16384
    ).to(device)

    optimizer = optim.AdamW(moco_model.parameters(), lr=learning_rate, weight_decay=1e-4)
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=1000)
    cosine = CosineAnnealingLR(optimizer, T_max=num_epochs - 1)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[1])

    start_epoch = 0
    best_loss = float('inf')

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print("Loading from checkpoint...")
        moco_model, optimizer, start_epoch, prev_loss, _ = load_checkpoint(resume_checkpoint, moco_model, optimizer, device)
        if prev_loss is not None:
            best_loss = prev_loss
        start_epoch += 1

    vision_student = torch.compile(vision_student)

    vision_augmenter = T.Compose([
        T.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.RandomAffine(degrees=1, translate=(0.02, 0.02))
    ])

    for epoch in range(start_epoch, num_epochs):
        moco_model.train()
        total_epoch_loss = 0.0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch_idx, batch_data in enumerate(progress_bar):
            images = batch_data['images'].to(device, non_blocking=True)
            audio_inputs = batch_data['spectrograms'].to(device, non_blocking=True)

            if moco_model.training:
                images = vision_augmenter(images)

                freq_mask = T_audio.FrequencyMasking(freq_mask_param=15)
                time_mask = T_audio.TimeMasking(time_mask_param=30)
                audio_inputs = torch.stack([time_mask(freq_mask(x)) for x in audio_inputs.unbind(0)])

            optimizer.zero_grad()

            loss = moco_model(images, audio_inputs)

            loss.backward()
            clip_grad_norm_(moco_model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            total_epoch_loss += loss.item()

            progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})
            if batch_idx % 50 == 0:
                print(f"  [Debug] Current Loss: {loss.item():.4f}")

        avg_loss = total_epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} | Avg Total Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

        if avg_loss < best_loss:
            print(f"*** New Best Loss: ({best_loss:.4f} -> {avg_loss:.4f}) ***")
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': moco_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, os.path.join(save_dir, "grandstaff_finetune_best.pth"))

        print("\n--- Phase 3 Validation Retrieval ---")

        metrics = evaluate_retrieval(
            moco_model.encoder_q_vision,
            moco_model.encoder_q_audio,
            val_loader,
            device
        )

        print("Audio-to-Vision:")
        print(f"  R@1: {metrics['A2V']['R@1']:.3f} | R@25: {metrics['A2V']['R@25']:.3f} | MRR: {metrics['A2V']['MRR']:.3f} | MR: {metrics['A2V']['MR']}")
        print("Vision-to-Audio:")
        print(f"  R@1: {metrics['V2A']['R@1']:.3f} | R@25: {metrics['V2A']['R@25']:.3f} | MRR: {metrics['V2A']['MRR']:.3f} | MR: {metrics['V2A']['MR']}")
        print("------------------------------------\n")
    return moco_model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


audio_path = hf_hub_download(repo_id="hyg444/SheetAudio-GCL", filename="SheetAudio-GCL-Audio.pth")
vision_path = hf_hub_download(repo_id="hyg444/SheetAudio-GCL", filename="SheetAudio-GCL-Sheet.pth")
filterbank_path = hf_hub_download(repo_id="hyg444/SheetAudio-GCL", filename="madmom_filterbank.pt")

vision_encoder = SheetMusicSwin().to(device)
audio_encoder = SpectrogramSwin().to(device)

vision_encoder.load_state_dict(torch.load(vision_path, map_location=device))
audio_encoder.load_state_dict(torch.load(audio_path, map_location=device))

filterbank = torch.load(filterbank_path, map_location='cpu')
audio_pipeline = MSMDSpectrogramPipeline(filterbank=filterbank)
audio_pipeline.eval()

hf_train = load_dataset('parquet', data_files={'train': 'hf://datasets/PRAIG/grandstaff-grandstaff-multimodal/data/train-*.parquet'}, split='train')
hf_val = load_dataset('parquet', data_files={'val': 'hf://datasets/PRAIG/grandstaff-grandstaff-multimodal/data/val-*.parquet'}, split='val')

train_dataset = GrandstaffFinetuneDataset(hf_dataset=hf_train, audio_pipeline=audio_pipeline, mode='train')
val_dataset = GrandstaffFinetuneDataset(hf_dataset=hf_val, audio_pipeline=audio_pipeline, mode='val')

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=grandstaff_collate_fn,
    drop_last=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=grandstaff_collate_fn,
    num_workers=2
)

save_directory = '/content/drive/MyDrive/MSMD_Checkpoints'
finetune_checkpoint = os.path.join(save_directory, 'grandstaff_finetune_best.pth')

moco_model = train_phase_3(
    train_loader=train_loader,
    val_loader=val_loader,
    vision_student=vision_encoder,
    audio_teacher=audio_encoder,
    num_epochs=15,
    learning_rate=1e-4,
    save_dir=save_directory,
    resume_checkpoint=finetune_checkpoint,
    device=device
)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

###Evaluate on Grandstaff


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# fine-tuned checkpoint on Google Drive
checkpoint_path = '/content/drive/MyDrive/MSMD_Checkpoints/grandstaff_finetune_best.pth'

print("Initializing models...")
vision_base = SheetMusicSwin().to(device)
audio_base = SpectrogramSwin().to(device)

moco_model = VisionAudioMoCo(
    vision_encoder=vision_base,
    audio_encoder=audio_base,
    dim=512,
    K=16384
).to(device)

print(f"Loading fine-tuned weights from: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location=device)
moco_model.load_state_dict(checkpoint['model_state_dict'])

finetuned_vision_encoder = moco_model.encoder_q_vision
finetuned_audio_encoder = moco_model.encoder_q_audio

finetuned_vision_encoder.eval()
finetuned_audio_encoder.eval()

print("\n--- Evaluating on PRAIG/grandstaff-grandstaff-multimodal (Test Split) ---")

hf_dataset = load_dataset(
    'parquet',
    data_files={'test': 'hf://datasets/PRAIG/grandstaff-grandstaff-multimodal/data/test-*.parquet'},
    split='test'
)

grandstaff_dataset = GrandstaffFinetuneDataset(
    hf_dataset=hf_dataset,
    audio_pipeline=audio_pipeline,
    mode='val'
)

BATCH_SIZE = 32
grandstaff_loader = DataLoader(
    grandstaff_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=grandstaff_collate_fn,
    num_workers=2
)

grandstaff_metrics = evaluate_retrieval(
    finetuned_vision_encoder,
    finetuned_audio_encoder,
    grandstaff_loader,
    device
)

print("\n--- Final Fine-Tuned Results ---")
print("Audio-to-Vision:")
print(f"  R@1: {grandstaff_metrics['A2V']['R@1']:.3f} | R@25: {grandstaff_metrics['A2V']['R@25']:.3f} | MRR: {grandstaff_metrics['A2V']['MRR']:.3f} | MR: {grandstaff_metrics['A2V']['MR']}")
print("Vision-to-Audio:")
print(f"  R@1: {grandstaff_metrics['V2A']['R@1']:.3f} | R@25: {grandstaff_metrics['V2A']['R@25']:.3f} | MRR: {grandstaff_metrics['V2A']['MRR']:.3f} | MR: {grandstaff_metrics['V2A']['MR']}")